# Norway/Base/Smiley Four-Panel Plot\n\nLoads saved tensors and draws the requested four panels. Tweak the plotting knobs in the second cell.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from data import COLOR_NAMES, NUM_CLASSES
from reward import norwegian_region_scores
from reward_smiley import smiley_scores

DATA_PATH = Path("figures/norway_base_smiley_four_panel_data.pt")
OUT_PATH = Path("figures/norway_base_smiley_four_panel_tweak.png")

bundle = torch.load(DATA_PATH, map_location="cpu")
panels = bundle["panels"]
print(bundle["description"])
print({k: tuple(v.shape) for k, v in panels.items()})
print(bundle["sources"])

In [ ]:
# Plot knobs.
PLOT_COLORS = {
    "blue": "#006DCC",
    "white": "#F8F8F8",
    "red": "#FF2D20",
}
TITLES = ["Prior", "Base", "Norway", "Smiley"]
PANEL_KEYS = ["prior", "base", "norway", "smiley"]
PANEL_KINDS = ["prior", "base", "norway", "smiley"]
LIMITS = {
    "prior": (-3.0, 3.0),
    "base": (-1.08, 1.08),
    "norway": (-1.08, 1.08),
    "smiley": (-1.08, 1.08),
}
MARKER_SIZE = {
    "prior": 3.2,
    "base": 4.8,
    "norway": 4.8,
    "smiley": 8.5,
}
ALPHA = 0.90
FIGSIZE = (13.0, 3.35)
DPI = 220
WSPACE = 0.045


def panel_target(x, kind):
    y = x[:, NUM_CLASSES:]
    if kind == "norway":
        scores = norwegian_region_scores(y[:, 0], y[:, 1], order="model")
        return scores.argmax(dim=-1), scores.max(dim=-1).values
    if kind == "smiley":
        scores = smiley_scores(y[:, 0], y[:, 1])
        return scores.argmax(dim=-1), scores.max(dim=-1).values
    return None, torch.zeros(x.shape[0])


def plot_panel(ax, x, title, kind):
    c = x[:, :NUM_CLASSES]
    y = x[:, NUM_CLASSES:]
    pred = c.argmax(dim=-1)

    target, strength = panel_target(x, kind)
    if kind == "smiley":
        # Draw generic/incorrect smiley points first. Then draw correct target
        # colors on top, with blue right-eye points and white mouth points last.
        correct = pred == target
        blue_right_eye = correct & (target == 0)
        white_mouth = correct & (target == 1)
        red_left_eye = correct & (target == 2)
        priority = (
            strength
            + 10.0 * red_left_eye.float()
            + 20.0 * blue_right_eye.float()
            + 30.0 * white_mouth.float()
        )
        order = torch.argsort(priority)
    else:
        order = torch.arange(x.shape[0])

    y_np = y[order].numpy()
    pred_np = pred[order].numpy()
    for color_idx, name in enumerate(COLOR_NAMES):
        mask = pred_np == color_idx
        ax.scatter(
            y_np[mask, 0],
            y_np[mask, 1],
            s=MARKER_SIZE[kind],
            c=PLOT_COLORS[name],
            alpha=ALPHA,
            edgecolors="#AFAFAF" if name == "white" else "none",
            linewidths=0.08 if name == "white" else 0.0,
            rasterized=True,
        )

    ax.set_facecolor("#050505")
    ax.set_xlim(*LIMITS[kind])
    ax.set_ylim(*LIMITS[kind])
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=14, pad=5)
    ax.set_xlabel(r"$y_1$", fontsize=13, labelpad=0)
    ax.set_ylabel(r"$y_2$", fontsize=13, labelpad=0)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)


fig, axes = plt.subplots(1, 4, figsize=FIGSIZE, constrained_layout=False)
for ax, key, title, kind in zip(axes, PANEL_KEYS, TITLES, PANEL_KINDS):
    plot_panel(ax, panels[key], title, kind)

fig.subplots_adjust(left=0.018, right=0.996, bottom=0.13, top=0.88, wspace=WSPACE)
fig.savefig(OUT_PATH, dpi=DPI)
OUT_PATH